# DNABERT2 Shared Split Promoter Benchmark

This notebook runs DNABERT2 on the same GSE144621 train/validation/test CSV split used by the CNN benchmark.

Scientific rule: train on `train`, select the threshold on `validation` using MCC, and report final performance on held-out `test`. Do not tune on the test split.

## What Is Different From The CNN Notebooks?

- CNN uses one-hot DNA sequence tensors.
- DNABERT2 uses a pretrained genomic language model tokenizer/encoder.
- The frozen DNABERT2 run caches embeddings and trains only a classifier head.
- The optional fine-tuning run updates DNABERT2 weights and is more compute-heavy.

The dataset split, seed, threshold rule, and metrics stay the same so model comparisons remain fair. The learning rate and batch size are model-specific when full DNABERT2 fine-tuning is used because CNN and DNABERT2 optimize very different parameter sets.

## What Was Checked From The Original DNABERT Notebooks?

The original `dna-bert/` notebooks on `dev` were inspected before making this benchmark notebook. They confirm the intended DNABERT2 model (`zhihan1996/DNABERT-2-117M`), `model_max_length` values around 75-100, learning rates around `3e-5` for full fine-tuning, and evaluation by Matthews correlation in the larger script run. However, the committed outputs there are not directly comparable to the CNN benchmark because some cells use sample data, some use regression/RMSE, and some use ad hoc splits.

This notebook therefore keeps the useful DNABERT2 settings but forces the same promoter CSV split and binary classification metrics used by CNN.


In [ ]:
# Colab setup
# Use Runtime > Change runtime type > GPU before running DNABERT2.

from pathlib import Path
import os
import sys

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
REPO_URL = "https://github.com/simplyshree/SeqTrainer.git"
BRANCH = "issue-3-cnn-baseline-reproduction"
REPO_DIR = Path("/content/SeqTrainer") if IN_COLAB else Path.cwd()

if IN_COLAB and not REPO_DIR.exists():
    clone_status = os.system(f"git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}")
    if clone_status != 0:
        raise RuntimeError(f"Could not clone branch {BRANCH}. Check that the branch exists on {REPO_URL}.")

if not REPO_DIR.exists():
    raise FileNotFoundError(f"Repository directory was not created: {REPO_DIR}")

os.chdir(REPO_DIR)

if IN_COLAB:
    # Colab keeps /content/SeqTrainer if you rerun cells without factory-resetting the runtime.
    # Always update to the latest pushed benchmark branch so package fixes are picked up.
    update_status = os.system(f"git fetch origin {BRANCH} && git checkout {BRANCH} && git reset --hard origin/{BRANCH}")
    if update_status != 0:
        raise RuntimeError(f"Could not update {REPO_DIR} to origin/{BRANCH}.")

print("Repository:", REPO_DIR)
!git rev-parse --abbrev-ref HEAD
!git rev-parse HEAD


In [ ]:
# Install package dependencies.
# `accelerate`, `einops`, and `triton` are included because DNABERT2 remote code commonly needs them in Colab.

if IN_COLAB:
    !python -m pip install -q --upgrade pip setuptools wheel
    !python -m pip install -q -e ".[torch]" accelerate einops triton

import numpy as np
import pandas as pd
import torch

print("python:", sys.version)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

In [ ]:
# Prepare the shared promoter CSV split.
# Default: use the bundled repo ZIP so Colab does not depend on Google Drive.
# Set USE_GOOGLE_DRIVE = True only when you specifically want to copy files from Drive.

from pathlib import Path
import shutil
import zipfile

USE_GOOGLE_DRIVE = False
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1rH47oJEjQjkJvHXKX_rwDjDb--dGPGx2"
DRIVE_FILE_IDS = {
    "train_EP_DNA_BERT2_genomic_order.csv": "1dOJr48-cyYWUL5A6Dgs7FEuImgApIrW6",
    "eval_EP_DNA_BERT2_genomic_order.csv": "16Hs7j4FznJQAXVoCxCWhO_A6CE0lTNtF",
    "test_EP_DNA_BERT2_genomic_order.csv": "1gIxdpwaZPvAwMFJ2c6LSl-J5FHrdPHmU",
}

DATA_DIR = REPO_DIR / "data" / "promoter_classification"
DATA_DIR.mkdir(parents=True, exist_ok=True)

split_file_names = {
    "train": "train_EP_DNA_BERT2_genomic_order.csv",
    "validation": "eval_EP_DNA_BERT2_genomic_order.csv",
    "test": "test_EP_DNA_BERT2_genomic_order.csv",
}

def files_ready():
    missing = [name for name in split_file_names.values() if not (DATA_DIR / name).exists()]
    if missing:
        print("Missing split files:", missing)
        return False
    return True

def extract_from_repo_zip():
    zip_path = REPO_DIR / "data" / "data_DNABERT" / "promoter_classification_DNABERT.zip"
    if not zip_path.exists():
        print("Repo ZIP not found:", zip_path)
        return False
    with zipfile.ZipFile(zip_path) as zf:
        archive_names = set(zf.namelist())
        for file_name in split_file_names.values():
            matches = [name for name in archive_names if Path(name).name == file_name and not name.startswith("__MACOSX/")]
            if not matches:
                print(f"{file_name} is missing from {zip_path}")
                return False
            with zf.open(matches[0]) as src, (DATA_DIR / file_name).open("wb") as dst:
                shutil.copyfileobj(src, dst)
    print("Extracted split files from repo ZIP:", zip_path)
    return True

def try_mount_drive():
    if not IN_COLAB:
        return None
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False, timeout_ms=120000)
        return Path("/content/drive/MyDrive")
    except Exception as exc:
        print("Drive mount skipped or failed. Continuing without Drive.")
        print(type(exc).__name__, exc)
        return None

def copy_from_drive(my_drive):
    if my_drive is None:
        return False
    candidates = [
        my_drive / "AIxBio" / "Promoter Classification" / "Data",
        my_drive / "AI BIO" / "Promoter Classification" / "Data",
        my_drive / "AI*BIO" / "Promoter Classification" / "Data",
    ]
    for drive_dir in candidates:
        if all((drive_dir / name).exists() for name in split_file_names.values()):
            for file_name in split_file_names.values():
                shutil.copy2(drive_dir / file_name, DATA_DIR / file_name)
            print("Copied split files from Drive:", drive_dir)
            return True
    print("Drive mounted, but expected files were not found in the usual folders.")
    return False

def _ensure_gdown():
    try:
        import gdown
        return gdown
    except ModuleNotFoundError:
        print("Installing gdown for direct Drive file download...")
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
        import gdown
        return gdown

def download_public_drive_files():
    if not IN_COLAB:
        return False
    gdown = _ensure_gdown()
    ok = True
    for file_name, file_id in DRIVE_FILE_IDS.items():
        target = DATA_DIR / file_name
        if target.exists():
            continue
        try:
            url = f"https://drive.google.com/uc?id={file_id}"
            gdown.download(url, str(target), quiet=False, fuzzy=True, use_cookies=False)
        except Exception as exc:
            print(f"Direct Drive download failed for {file_name}.")
            print(type(exc).__name__, exc)
            ok = False
    if ok and files_ready():
        print("Downloaded split files directly from Drive file IDs:", DRIVE_FOLDER_URL)
        return True
    return False

if not files_ready():
    prepared = extract_from_repo_zip()
    if not prepared and USE_GOOGLE_DRIVE:
        prepared = copy_from_drive(try_mount_drive())
    if not prepared:
        prepared = download_public_drive_files()

if not files_ready():
    raise FileNotFoundError(
        f"Could not prepare all split files in {DATA_DIR}. "
        "Use the bundled ZIP, set USE_GOOGLE_DRIVE=True, or manually upload the three CSV files."
    )

for split, file_name in split_file_names.items():
    path = DATA_DIR / file_name
    df = pd.read_csv(path)
    print(split, path, df.shape)
    print(df["label"].value_counts().sort_index().to_dict())


In [ ]:
# Create Colab configs that allow Hugging Face model download.
# The committed configs keep downloads off by default for tests and offline reproducibility.

COLAB_CONFIG_DIR = REPO_DIR / "outputs" / "colab_configs"
COLAB_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

def make_colab_config(source_config, output_dir):
    src = REPO_DIR / source_config
    text = src.read_text(encoding="utf-8")
    if "allow_download" in text:
        text = text.replace("allow_download = false", "allow_download = true")
    else:
        text = text.replace("[model.params]\n", "[model.params]\nallow_download = true\n", 1)
    lines = []
    for line in text.splitlines():
        if line.startswith('output_dir = '):
            lines.append(f'output_dir = "{output_dir.as_posix()}"')
        else:
            lines.append(line)
    text = "\n".join(lines) + "\n"
    target = COLAB_CONFIG_DIR / Path(source_config).name
    target.write_text(text, encoding="utf-8")
    return target

FROZEN_CONFIG = make_colab_config(
    "config-examples/benchmarks/dnabert2_frozen.toml",
    Path("outputs/benchmarks/dnabert2_frozen_colab"),
)
FINETUNE_CONFIG = make_colab_config(
    "config-examples/benchmarks/dnabert2_finetune.toml",
    Path("outputs/benchmarks/dnabert2_finetune_colab"),
)

print("Frozen config:", FROZEN_CONFIG)
print("Fine-tune config:", FINETUNE_CONFIG)

In [ ]:
# Run frozen DNABERT2 embedding benchmark.
# This is the first DNABERT2 result to compare against CNN-v2.

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from seqtrainer.benchmarks.runner import run_benchmark

frozen_result = run_benchmark(FROZEN_CONFIG, base_dir=REPO_DIR, allow_skip=False)
print("status:", frozen_result.status)
print("output_dir:", frozen_result.output_dir)

In [ ]:
# Inspect frozen DNABERT2 metrics.

frozen_metrics = pd.read_csv(frozen_result.output_dir / "metrics.csv")
display(frozen_metrics)

history_path = frozen_result.output_dir / "history.csv"
if history_path.exists():
    history = pd.read_csv(history_path)
    display(history.tail())

print("Use validation MCC for model selection, then read the test row for final reporting.")

In [ ]:
# Optional: run full DNABERT2 fine-tuning.
# Keep this False until the frozen run completes and you have enough GPU time.

RUN_FINE_TUNE = False

if RUN_FINE_TUNE:
    finetune_result = run_benchmark(FINETUNE_CONFIG, base_dir=REPO_DIR, allow_skip=False)
    print("status:", finetune_result.status)
    print("output_dir:", finetune_result.output_dir)
    display(pd.read_csv(finetune_result.output_dir / "metrics.csv"))
    hist = pd.read_csv(finetune_result.output_dir / "history.csv")
    display(hist.tail())
else:
    print("Fine-tuning skipped. Set RUN_FINE_TUNE = True after frozen DNABERT2 is working.")

In [ ]:
# Optional comparison helper.
# Add CNN output folders here if you copied or generated them in the same runtime.

from seqtrainer.benchmarks.compare import compare_benchmark_runs

candidate_dirs = [frozen_result.output_dir]
if "finetune_result" in globals():
    candidate_dirs.append(finetune_result.output_dir)

comparison_dir = REPO_DIR / "outputs" / "benchmarks" / "dnabert2_colab_comparison"
comparison = compare_benchmark_runs(candidate_dirs, output_dir=comparison_dir)
print("comparison_dir:", comparison.output_dir)
display(pd.read_csv(comparison.output_dir / "comparison_metrics.csv"))